In [1]:
%run ~/drg-pipeline/data-cleaning/drg-cleaning-v2.ipynb

SyntaxError: invalid syntax (3398668055.py, line 1)

SyntaxError: invalid syntax (3398668055.py, line 1)

In [ ]:
# Define paths
paths <- list(
  year_to_load = here::here("data-cleaning", "cache", "year_to_load.txt"),
  input_notebook = here::here("data-cleaning", "drg-cleaning-v2.ipynb"),
  output_rscript = here::here("data-cleaning", "debug", "drg-cleaning.r")
)

# Helper function to run the notebook as an R script with enhanced logging
run_notebook <- function(to_parallel) {
  # Set the environment variable for parallel execution
  Sys.setenv(TO_PARALLEL = to_parallel)

  # Convert the Jupyter notebook to an R script
  nbconvert_status <- system(paste(
    "jupyter nbconvert --no-prompt --to script",
    paths$input_notebook, "--output", paths$output_rscript
  ), intern = TRUE)

  # Print any conversion messages or errors
  cat("Nbconvert output:\n", nbconvert_status, "\n")

  # Run the generated R script and capture its output
  rscript_status <- system(paste("Rscript", paths$output_rscript), intern = TRUE)

  # Print the R script output to see any issues or logs
  cat("Rscript output:\n", rscript_status, "\n")

  # Check for any errors in the R script output and return FALSE if found
  if (any(grepl("Error", rscript_status, ignore.case = TRUE))) {
    return(FALSE)
  }
  return(TRUE)
}

# Loop over years (you can extend to multiple years)
for (year in c(2022)) {
  # Write the current year to the year_to_load file
  write(as.character(year), paths$year_to_load)

  # Try running the notebook with parallel execution
  if (!run_notebook(TRUE)) {
    message("Retrying with to_parallel = FALSE...")
    # If it fails, retry without parallel execution
    if (!run_notebook(FALSE)) {
      stop("Notebook execution failed with both to_parallel = TRUE and FALSE")
    }
  }
}
